In [ ]:
import os
import sys
#sys.path.insert(1, os.path.abspath(os.path.join(os.getcwd(), os.pardir)))
os.chdir('../')
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import LogLocator, ScalarFormatter, FixedFormatter

from woodtapper.example_sampling import RandomForestClassifierExplained

## Generate data set:

In [ ]:
output_dir_data_set = "reproduce-exp/sim-data-time/" #simulation data for timing
os.makedirs(output_dir_data_set, exist_ok=True)

In [ ]:
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split

In [ ]:
X_sim, y_sim = make_classification(n_samples=625000, n_features=200, n_informative=196, n_redundant=2, n_repeated=2,
                                   n_classes=2, n_clusters_per_class=2, weights=None, flip_y=0.01, class_sep=1.0,
                                   random_state=42)
X_train_original, X_test_original, y_train_original, y_test_original = train_test_split(X_sim, y_sim, test_size=0.2, random_state=42)

In [ ]:
np.savetxt(f"{output_dir_data_set}/X_train.csv", X_train_original, delimiter = ",")
np.savetxt(f"{output_dir_data_set}/y_train.csv", y_train_original, delimiter = ",")
np.savetxt(f"{output_dir_data_set}/X_test.csv", X_test_original, delimiter = ",")
np.savetxt(f"{output_dir_data_set}/y_test.csv", y_test_original, delimiter = ",")

In [ ]:
X_train_original = np.genfromtxt(f"{output_dir_data_set}/X_train.csv", delimiter = ",")
y_train_original = np.genfromtxt(f"{output_dir_data_set}/y_train.csv", delimiter = ",")
X_test_original = np.genfromtxt(f"{output_dir_data_set}/X_test.csv", delimiter = ",")
y_test_original = np.genfromtxt(f"{output_dir_data_set}/y_test.csv", delimiter = ",")

In [ ]:
output_dir_ExSampling = "reproduce-exp/time-ExSampling/" #simulation data for example sampling
os.makedirs(output_dir_ExSampling, exist_ok=True)

## Timing by samples:

In [ ]:
n_samples_train = 500
n_samples_test = 100000
n_dim = 10

X_train = X_train_original[:n_samples_train,:n_dim]
y_train = y_train_original[:n_samples_train]

X_test = X_test_original[:n_samples_test,:n_dim] 
y_test = y_test_original[:n_samples_test]

In [ ]:
grf_clf = RandomForestClassifierExplained(n_estimators=100,max_features=10,n_jobs=1,random_state=0)
grf_clf.fit(X_train, y_train)

In [ ]:
list_n_samples_test = [1000, 2500, 5000, 7500, 10000, 25000, 50000, 75000, 100000]

In [ ]:
##### SAMPLES ####
list_time_samples_1 = []  
for n_samples_test in list_n_samples_test:
    X_test_subset = X_test_original[:n_samples_test,:n_dim]
    time_start = time.time()
    #weights = grf_clf.get_weights(X_test_subset)
    weights = grf_clf.get_weights_cython(X_test_subset)
    time_end = time.time()
    time_ = time_end - time_start
    list_time_samples_1.append(time_)
    print("Get kernel time for n_samples_test =", n_samples_test, ": ", time_) 

list_time_samples_1

In [ ]:
##### SAMPLES 2 ####
list_time_samples_2 = []  
for n_samples_test in list_n_samples_test:
    X_test_subset = X_test_original[:n_samples_test,:n_dim]
    time_start = time.time()
    weights = grf_clf.get_weights_cython(X_test_subset)
    time_end = time.time()
    time_ = time_end - time_start
    list_time_samples_2.append(time_)
    print("Get kernel time for n_samples_test =", n_samples_test, ": ", time_) 

list_time_samples_2

## Timing by dimension

In [ ]:
list_n_dims = [15, 25, 50, 75, 100, 125, 150, 175, 200] #20

In [ ]:
### DIMENSION ####
n_samples_train = 500
n_samples_test = 100000 #[1000, 2500, 5000, 7500, 10000, 25000, 50000, 75000, 100000]
list_time_dim_1 = []  
for n_dim in list_n_dims:
    X_train = X_train_original[:n_samples_train,:n_dim]
    y_train = y_train_original[:n_samples_train]
    grf_clf = RandomForestClassifierExplained(n_estimators=100,max_features=10,n_jobs=1,random_state=0)
    time_start = time.time()
    grf_clf.fit(X_train, y_train)
    time_end = time.time()
    print("Fit time: ", time_end - time_start)

    X_test_subset = X_test_original[:n_samples_test,:n_dim]
    time_start = time.time()
    weights = grf_clf.get_weights_cython(X_test_subset)
    time_end = time.time()
    time_ = time_end - time_start
    list_time_dim_1.append(time_)
    print("Get kernel time for n_samples_test =", n_samples_test, ": ", time_) 
list_time_dim_1

In [ ]:
### DIMENSION  2 ####
n_samples_train = 500
n_samples_test = 100000 #[1000, 2500, 5000, 7500, 10000, 25000, 50000, 75000, 100000]
list_time_dim_2 = []  
for n_dim in list_n_dims:
    X_train = X_train_original[:n_samples_train,:n_dim]
    y_train = y_train_original[:n_samples_train]
    grf_clf = RandomForestClassifierExplained(n_estimators=100,max_features=10,n_jobs=1,random_state=0)
    time_start = time.time()
    grf_clf.fit(X_train, y_train)
    time_end = time.time()
    print("Fit time: ", time_end - time_start)

    X_test_subset = X_test_original[:n_samples_test,:n_dim]
    time_start = time.time()
    weights = grf_clf.get_weights_cython(X_test_subset)
    time_end = time.time()
    time_ = time_end - time_start
    list_time_dim_2.append(time_)
    print("Get kernel time for n_samples_test =", n_samples_test, ": ", time_) 
list_time_dim_2

## Plot:

WARNING: Before running the following chunks, make sur you have already ran all the chunks from timing-skgr.py

In [ ]:
# SAMPLES dataframe #
list_n_samples_test = [1000, 2500, 5000, 7500, 10000, 25000, 50000, 75000, 100000]
res_time = pd.DataFrame({'n_samples_test': list_n_samples_test, 'time_1': list_time_samples_1, 'time_2': list_time_samples_2})
res_time["mean"] = res_time[["time_1","time_2"]].mean(axis=1)
res_time["std"] = res_time[["time_1","time_2"]].std(axis=1)
res_time


list_time_skgrf_1 = np.load(os.path.join(output_dir_ExSampling, "list_time_samples_skgrf_1.npy"))
list_time_skgrf_2 = np.load(os.path.join(output_dir_ExSampling, "list_time_samples_skgrf_2.npy"))
res_time_skgrf = pd.DataFrame({'n_samples_test': list_n_samples_test, 'time_1': list_time_skgrf_1, 'time_2': list_time_skgrf_2})
res_time_skgrf["mean"] = res_time_skgrf[["time_1","time_2"]].mean(axis=1)
res_time_skgrf["std"] = res_time_skgrf[["time_1","time_2"]].std(axis=1)
res_time_skgrf

In [ ]:
# DIMENSION dataframe #
res_time_dim = pd.DataFrame({'n_dim': list_n_dims, 'time_1': list_time_dim_1, 'time_2': list_time_dim_2})
res_time_dim["mean"] = res_time_dim[["time_1","time_2"]].mean(axis=1)
res_time_dim["std"] = res_time_dim[["time_1","time_2"]].std(axis=1)
res_time_dim

list_time_dim_skgrf_1 = np.load(os.path.join(output_dir_ExSampling, "list_time_dim_skgrf_1.npy"))
list_time_dim_skgrf_2 = np.load(os.path.join(output_dir_ExSampling, "list_time_dim_skgrf_2.npy"))
res_time_skgrf_dim = pd.DataFrame({'n_dim': list_n_dims, 'time_1': list_time_dim_skgrf_1, 'time_2': list_time_dim_skgrf_2})
res_time_skgrf_dim["mean"] = res_time_skgrf_dim[["time_1","time_2"]].mean(axis=1)
res_time_skgrf_dim["std"] = res_time_skgrf_dim[["time_1","time_2"]].std(axis=1)
res_time_skgrf_dim

In [ ]:
## Plotting ###
###############

# Create 1x3 subplot layout
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
#fig.suptitle(r"Computation Time for $n$=300K and $M$=1000", fontsize=20)


# --- Subplot 1: Fit Time ---
axes[0].plot(res_time["n_samples_test"].values, res_time["mean"].values, 'o--', label='Ours',c='royalblue')
axes[0].fill_between(res_time["n_samples_test"].values, res_time["mean"].values-res_time["std"].values,
                     res_time["mean"].values+res_time["std"].values,alpha=0.5,color='skyblue')
axes[0].plot(res_time["n_samples_test"].values, res_time_skgrf["mean"].values, 's--', label='skgrf',c='darkviolet')
axes[0].fill_between(res_time["n_samples_test"].values, res_time_skgrf["mean"].values-res_time_skgrf["std"].values,
                     res_time_skgrf["mean"].values+res_time_skgrf["std"].values,alpha=0.5,color='thistle')
axes[0].set_yscale('log')

# --- Y-axis log scale with powers of 10 ---
axes[0].set_yscale('log')
axes[0].yaxis.set_major_locator(LogLocator(base=10.0))       # ticks at 10^n
axes[0].yaxis.set_major_formatter(ScalarFormatter())          # show as 10^n
axes[0].yaxis.get_offset_text().set_visible(False)            # remove offset text
# --- X-axis linear but show powers of 10 ---
#x_ticks = [2e4, 4e4, 6e4, 8e4, 10e4]                      # linear spacing
#axes[0].set_xticks(x_ticks)
#axes[0].set_xticklabels([f'${i%1e4}$'+r"$\times$"+f'$10^{int(np.log10(tick))}$' for i,tick in enumerate(x_ticks)])  # show powers

axes[0].set_title("Weights computation",fontsize=18)
axes[0].set_xlabel("Number of test samples",fontsize=15)
axes[0].set_ylabel("Time (s)",fontsize=15)
axes[0].legend(fontsize=12)
axes[0].grid(False)

# --- Subplot 2: Rules Extraction ---
axes[1].plot(res_time_dim["n_dim"].values, res_time_dim["mean"].values, 'o--', label='Ours',c='royalblue')
axes[1].fill_between(res_time_dim["n_dim"].values, res_time_dim["mean"].values-res_time_dim["std"].values,
                     res_time_dim["mean"].values+res_time_dim["std"].values,alpha=0.5,color='skyblue')
axes[1].plot(res_time_dim["n_dim"].values, res_time_skgrf_dim["mean"].values, 's--', label='skgrf',c='darkviolet')
axes[1].fill_between(res_time_dim["n_dim"].values, res_time_skgrf_dim["mean"].values-res_time_skgrf_dim["std"].values,
                     res_time_skgrf_dim["mean"].values+res_time_skgrf_dim["std"].values,alpha=0.5,color='thistle')
axes[1].set_yscale('log')

# --- Y-axis log scale with powers of 10 ---
axes[1].set_yscale('log')
axes[1].yaxis.set_major_locator(LogLocator(base=10.0))       # ticks at 10^n
axes[1].yaxis.set_major_formatter(ScalarFormatter())          # show as 10^n
axes[1].yaxis.get_offset_text().set_visible(False)            # remove offset text
# --- X-axis linear but show powers of 10 ---
#x_ticks = [2e4, 4e4, 6e4, 8e4, 10e4]                      # linear spacing
#axes[1].set_xticks(x_ticks)
#axes[1].set_xticklabels([f'${i%1e4}$'+r"$\times$"+f'$10^{int(np.log10(tick))}$' for i,tick in enumerate(x_ticks)])  # show powers

axes[1].set_title("Weights computation",fontsize=18)
axes[1].set_xlabel("Number of dimensions",fontsize=15)
axes[1].set_ylabel("Time (s)",fontsize=15)
axes[1].legend(fontsize=12)
axes[1].grid(False)


plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.savefig(os.path.join(output_dir_ExSampling, "run-time-grf-log.pdf"))
plt.show()